In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read- only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/623.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/764.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/1075.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/771.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/208.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/820.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/473.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/1031.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/333.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/1024.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/537.jpg
/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian/N/45.jpg
/kaggle/input/datasets/prathumarikeri/

In [2]:
DATASET_PATH = "/kaggle/input/datasets/prathumarikeri/indian-sign-language-isl/Indian"

In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models

2026-04-24 17:06:15.207446: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777050375.691391      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777050375.810707      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777050377.030520      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777050377.030581      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777050377.030583      23 computation_placer.cc:177] computation placer alr

In [4]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

In [5]:
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 34198 images belonging to 35 classes.
Found 8547 images belonging to 35 classes.


In [6]:
print("Classes:", train_data.class_indices)
print("Total images:", train_data.samples)

Classes: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, 'A': 9, 'B': 10, 'C': 11, 'D': 12, 'E': 13, 'F': 14, 'G': 15, 'H': 16, 'I': 17, 'J': 18, 'K': 19, 'L': 20, 'M': 21, 'N': 22, 'O': 23, 'P': 24, 'Q': 25, 'R': 26, 'S': 27, 'T': 28, 'U': 29, 'V': 30, 'W': 31, 'X': 32, 'Y': 33, 'Z': 34}
Total images: 34198


In [7]:
base_model = InceptionV3(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

I0000 00:00:1777050437.788773      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777050437.794989      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1777050452.231734      86 service.cc:152] XLA service 0x7e50cc4177d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777050452.231779      86 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777050452.231783      86 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777050454.979960      86 cuda_dnn.cc:529] Loaded cuDNN version 91002


   2/1069 ━━━━━━━━━━━━━━━━━━━━ 1:30 85ms/step - accuracy: 0.0234 - loss: 4.7902      

I0000 00:00:1777050463.660501      86 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1069/1069 ━━━━━━━━━━━━━━━━━━━━ 904s 828ms/step - accuracy: 0.9192 - loss: 0.3688 - val_accuracy: 0.9888 - val_loss: 0.0602
Epoch 2/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 608s 569ms/step - accuracy: 0.9978 - loss: 0.0153 - val_accuracy: 0.9887 - val_loss: 0.0699
Epoch 3/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 609s 569ms/step - accuracy: 0.9978 - loss: 0.0139 - val_accuracy: 0.9899 - val_loss: 0.0625
Epoch 4/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 617s 578ms/step - accuracy: 0.9976 - loss: 0.0118 - val_accuracy: 0.9878 - val_loss: 0.0835
Epoch 5/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 621s 581ms/step - accuracy: 0.9969 - loss: 0.0128 - val_accuracy: 0.9885 - val_loss: 0.1004
Epoch 6/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 606s 567ms/step - accuracy: 0.9972 - loss: 0.0132 - val_accuracy: 0.9891 - val_loss: 0.0536
Epoch 7/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 612s 572ms/step - accuracy: 0.9971 - loss: 0.0102 - val_accuracy: 0.9876 - val_loss: 0.1038
Epoch 8/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 610s 571ms/step - accuracy: 0.9

In [11]:
model.save("/kaggle/working/isl_model_inc.h5")

In [12]:
model.save("/kaggle/working/isl_model_inception.keras")